# Dia 4 — Azure Machine Learning: Command Job + MLflow
### Curso: Introduccion a Herramientas de Computo en la Nube · Azure

---

Este notebook sigue el tutorial oficial de Microsoft:
**Quickstart: Get started with Azure Machine Learning (SDK v2)**
https://learn.microsoft.com/en-us/azure/machine-learning/tutorial-azure-ml-in-a-day

**Dataset:** Default of Credit Card Clients (30,000 clientes, Taiwan)
- Fuente: UCI ML Repository, disponible en URL publica de Azure
- Target: `default payment next month` (0 = no default, 1 = default)
- Features: limite de credito, genero, educacion, estado civil, historial de pagos, montos

| Seccion | Contenido |
|---------|----------|
| 0 | Instalacion |
| 1 | Conectar al workspace |
| 2 | Crear el script de entrenamiento (main.py) |
| 3 | Crear el Compute Cluster |
| 4 | Configurar y lanzar el Command Job |
| 5 | Ver resultados en Azure ML Studio |
| 6 | Registrar y usar el modelo |

---

**Prerequisitos antes de ejecutar:**
- Workspace creado en `portal.azure.com` (ver instrucciones en el HTML del dia)
- Azure CLI instalado y sesion iniciada: `az login`
- Subscription ID, Resource Group y Workspace name a la mano

---
## Seccion 0 — Instalacion
Ejecuta una sola vez.

In [ ]:
%pip install azure-ai-ml azure-identity --quiet
print('Listo.')

---
## Seccion 1 — Conectar al workspace

`DefaultAzureCredential` busca credenciales en este orden:
1. Variables de entorno
2. Azure CLI (la forma que usamos — requiere `az login` previo)
3. Managed Identity

Donde encontrar los valores en Azure ML Studio:
- Clic en el nombre del workspace (esquina superior derecha del Studio)
- Copia Subscription ID, Resource Group y Workspace name

In [ ]:
SUBSCRIPTION   = "TU_SUBSCRIPTION_ID"
RESOURCE_GROUP = "TU_RESOURCE_GROUP"
WS_NAME        = "workspace-curso-azure"

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

ml_client = MLClient(
    credential          = credential,
    subscription_id     = SUBSCRIPTION,
    resource_group_name = RESOURCE_GROUP,
    workspace_name      = WS_NAME,
)

# Esta llamada verifica que la conexion funciona
ws = ml_client.workspaces.get(WS_NAME)
print(f"Conectado al workspace: {ws.name}")
print(f"  Location      : {ws.location}")
print(f"  Resource Group: {ws.resource_group}")

---
## Seccion 2 — Crear el script de entrenamiento

Usamos `%%writefile` para escribir `main.py` directamente desde el notebook.
Este script es el que Azure ML ejecutara en el cluster.

El script:
1. Recibe argumentos (ruta del CSV, hiperparametros, nombre del modelo)
2. Carga y divide los datos
3. Entrena un `GradientBoostingClassifier`
4. Loggea todo con MLflow (`autolog` captura params y metricas automaticamente)
5. Registra el modelo en el workspace

In [ ]:
import os

train_src_dir = "./src"
os.makedirs(train_src_dir, exist_ok=True)
print(f"Directorio creado: {train_src_dir}")

In [ ]:
%%writefile src/main.py
import os
import argparse
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


def main():
    # ── argumentos de entrada ──────────────────────────────────────────────────
    parser = argparse.ArgumentParser()
    parser.add_argument("--data",                 type=str,   help="ruta al CSV de entrada")
    parser.add_argument("--test_train_ratio",     type=float, default=0.25)
    parser.add_argument("--n_estimators",         type=int,   default=100)
    parser.add_argument("--learning_rate",        type=float, default=0.1)
    parser.add_argument("--registered_model_name",type=str,   help="nombre del modelo en el registry")
    args = parser.parse_args()

    # ── iniciar MLflow ─────────────────────────────────────────────────────────
    mlflow.start_run()
    mlflow.sklearn.autolog()   # loggea params, metricas y el modelo automaticamente

    # ── cargar datos ───────────────────────────────────────────────────────────
    print(f"Cargando datos desde: {args.data}")
    credit_df = pd.read_csv(args.data, header=1, index_col=0)

    mlflow.log_metric("num_samples",  credit_df.shape[0])
    mlflow.log_metric("num_features", credit_df.shape[1] - 1)

    print(f"Dataset cargado: {credit_df.shape[0]} filas, {credit_df.shape[1]} columnas")
    print(f"Distribucion del target:")
    print(credit_df["default payment next month"].value_counts())

    # ── dividir en train/test ──────────────────────────────────────────────────
    train_df, test_df = train_test_split(credit_df, test_size=args.test_train_ratio, random_state=42)

    y_train = train_df.pop("default payment next month")
    X_train = train_df.values

    y_test  = test_df.pop("default payment next month")
    X_test  = test_df.values

    print(f"Train: {X_train.shape} | Test: {X_test.shape}")

    # ── entrenar el modelo ─────────────────────────────────────────────────────
    clf = GradientBoostingClassifier(
        n_estimators  = args.n_estimators,
        learning_rate = args.learning_rate,
        random_state  = 42
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print("\nReporte de clasificacion:")
    print(classification_report(y_test, y_pred, target_names=["No default", "Default"]))

    # ── registrar el modelo ────────────────────────────────────────────────────
    print(f"Registrando modelo como: {args.registered_model_name}")

    conda_env = {
        'name': 'mlflow-env',
        'channels': ['conda-forge'],
        'dependencies': [
            'python=3.10.15',
            {'pip': [
                'mlflow==2.17.0',
                'pandas==1.5.3',
                'scikit-learn==1.5.2',
                'numpy==1.26.4',
            ]}
        ],
    }

    mlflow.sklearn.log_model(
        sk_model              = clf,
        registered_model_name = args.registered_model_name,
        artifact_path         = args.registered_model_name,
        conda_env             = conda_env,
    )

    mlflow.sklearn.save_model(
        sk_model = clf,
        path     = os.path.join(args.registered_model_name, "trained_model"),
    )

    mlflow.end_run()
    print("Job completado.")


if __name__ == "__main__":
    main()

In [ ]:
# Verificar que el script fue creado correctamente
with open("src/main.py") as f:
    lineas = f.readlines()
print(f"main.py creado: {len(lineas)} lineas")

---
## Seccion 3 — Crear el Compute Cluster

Un **Compute Cluster** es un conjunto de VMs que Azure ML administra automaticamente.
Con `min_instances=0`, el cluster se apaga cuando no hay trabajo y no genera costo.

Nota: tambien puedes usar **Serverless Compute** (sin configurar nada),
pero el cluster da mas control sobre el tipo de VM y el costo.

In [ ]:
from azure.ai.ml.entities import AmlCompute

COMPUTE_NAME = "cluster-curso"

try:
    cluster = ml_client.compute.get(COMPUTE_NAME)
    print(f"El cluster '{COMPUTE_NAME}' ya existe.")
    print(f"  Tipo  : {cluster.size}")
    print(f"  Estado: {cluster.provisioning_state}")
except:
    print(f"Creando cluster '{COMPUTE_NAME}'...")
    cluster = AmlCompute(
        name          = COMPUTE_NAME,
        type          = "amlcompute",
        size          = "Standard_DS2_v2",  # 2 vCPUs, 7 GB RAM
        min_instances = 0,                   # escala a 0 cuando esta inactivo
        max_instances = 2,
        idle_time_before_scale_down = 120    # apaga tras 2 min sin trabajo
    )
    poller  = ml_client.compute.begin_create_or_update(cluster)
    cluster = poller.result()
    print(f"Cluster creado. Estado: {cluster.provisioning_state}")

---
## Seccion 4 — Configurar y lanzar el Command Job

Un **Command Job** en Azure ML empaqueta:
- El codigo (`./src/`)
- El entorno Python (usamos un entorno curado de Azure ML con sklearn)
- Los inputs (el CSV desde URL publica)
- El comando a ejecutar (`python main.py ...`)

Azure ML corre todo esto en el cluster y guarda los logs, metricas y el modelo.

In [ ]:
from azure.ai.ml import command, Input

REGISTERED_MODEL_NAME = "credit_defaults_model"

job = command(
    # codigo fuente a empaquetar y enviar al cluster
    code    = "./src/",

    # comando que se ejecutara en el cluster
    command = (
        "python main.py "
        "--data ${{inputs.data}} "
        "--test_train_ratio ${{inputs.test_train_ratio}} "
        "--n_estimators ${{inputs.n_estimators}} "
        "--learning_rate ${{inputs.learning_rate}} "
        "--registered_model_name ${{inputs.registered_model_name}}"
    ),

    # inputs del job
    inputs = dict(
        data = Input(
            type = "uri_file",
            # CSV en URL publica de Azure — no necesitas subirlo
            path = "https://azuremlexamples.blob.core.windows.net/datasets/credit_card/default_of_credit_card_clients.csv",
        ),
        test_train_ratio      = 0.2,
        n_estimators          = 100,
        learning_rate         = 0.25,
        registered_model_name = REGISTERED_MODEL_NAME,
    ),

    # entorno Python curado por Azure ML con sklearn preinstalado
    environment  = "azureml://registries/azureml/environments/sklearn-1.5/labels/latest",

    # compute donde correra el job
    compute      = COMPUTE_NAME,

    # nombre que aparece en Azure ML Studio > Jobs
    display_name = "credit_default_prediction",

    # nombre del experimento que agrupa los runs
    experiment_name = "curso_azure_dia4_credit",
)

print("Command Job configurado:")
print(f"  Script       : src/main.py")
print(f"  Dataset      : credit_card_clients.csv (30,000 filas)")
print(f"  Algoritmo    : GradientBoostingClassifier")
print(f"  n_estimators : 100")
print(f"  learning_rate: 0.25")
print(f"  Compute      : {COMPUTE_NAME}")

In [ ]:
# Enviar el job al cluster
# Esta celda regresa inmediatamente — el job corre en background en Azure
returned_job = ml_client.create_or_update(job)

print(f"Job enviado al cluster.")
print(f"  Nombre       : {returned_job.name}")
print(f"  Display name : {returned_job.display_name}")
print(f"  Estado       : {returned_job.status}")
print()
print("Sigue el progreso en Azure ML Studio:")
print("  Jobs > curso_azure_dia4_credit > credit_default_prediction")
print()
print("Mientras el job corre, explora el portal:")
print("  - Tab 'Overview': estado, duracion, compute usado")
print("  - Tab 'Metrics' : aparece en tiempo real mientras entrena")
print("  - Tab 'Outputs' : logs del script (stdout de main.py)")

---
## Seccion 5 — Ver resultados en Azure ML Studio

Mientras el job corre (o una vez que termina), ve al portal y explora:

**Jobs > curso_azure_dia4_credit > credit_default_prediction**

- `Overview`: estado, duracion, tipo de compute y costo estimado
- `Metrics`: accuracy, precision, recall, AUC — loggeados por MLflow autolog
- `Outputs + logs`: el stdout completo de main.py, incluyendo el classification_report
- `Explanations`: si esta disponible, muestra feature importance

La siguiente celda espera a que el job termine para continuar.

In [ ]:
# Esperar a que el job termine (puede tardar 5-10 minutos)
# Esta celda bloquea hasta que el status sea Completed o Failed
ml_client.jobs.stream(returned_job.name)

job_final = ml_client.jobs.get(returned_job.name)
print(f"\nJob finalizado.")
print(f"  Estado: {job_final.status}")

In [ ]:
# Consultar las metricas loggeadas por MLflow
import mlflow

# Conectar MLflow al workspace de Azure ML
mlflow.set_tracking_uri(ml_client.workspaces.get(WS_NAME).mlflow_tracking_uri)

# Buscar el run correspondiente al job
runs = mlflow.search_runs(
    experiment_names=["curso_azure_dia4_credit"],
    order_by=["start_time DESC"]
)

if not runs.empty:
    ultimo_run = runs.iloc[0]
    print("Metricas del ultimo run:")
    metricas = [c for c in runs.columns if c.startswith("metrics.")]
    for m in metricas:
        if not pd.isna(ultimo_run[m]):
            print(f"  {m.replace('metrics.', ''):30s}: {ultimo_run[m]:.4f}")
else:
    print("No se encontraron runs. Verifica que el job haya terminado.")

---
## Seccion 6 — Registrar y usar el modelo

El script ya registro el modelo automaticamente en el Model Registry de Azure ML.
Aqui lo consultamos y lo cargamos para hacer predicciones locales.

In [ ]:
# Consultar el modelo registrado en Azure ML
modelos = list(ml_client.models.list(name=REGISTERED_MODEL_NAME))

if modelos:
    modelo_info = modelos[0]
    print(f"Modelo registrado en Azure ML:")
    print(f"  Nombre  : {modelo_info.name}")
    print(f"  Version : {modelo_info.version}")
    print(f"  Path    : {modelo_info.path}")
    print()
    print("Puedes verlo en Azure ML Studio > Models")
else:
    print(f"El modelo '{REGISTERED_MODEL_NAME}' no se encontro todavia.")
    print("Espera a que el job termine y vuelve a ejecutar esta celda.")

In [ ]:
import os

# Descargar el modelo localmente
download_path = "./modelo_descargado"
os.makedirs(download_path, exist_ok=True)

ml_client.models.download(
    name        = REGISTERED_MODEL_NAME,
    version     = modelos[0].version,
    download_path = download_path
)

print(f"Modelo descargado en: {download_path}")
for root, dirs, files in os.walk(download_path):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path}")

In [ ]:
import mlflow.sklearn
import pandas as pd
import numpy as np

# Cargar el modelo desde el directorio descargado
model_path = os.path.join(download_path, REGISTERED_MODEL_NAME, "trained_model")
modelo     = mlflow.sklearn.load_model(model_path)

print(f"Modelo cargado: {type(modelo).__name__}")
print()

# Datos de prueba — mismas 23 features que el dataset original
# (LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE, PAY_0..PAY_6, BILL_AMT1..6, PAY_AMT1..6)
# Simulamos 3 perfiles de clientes
perfiles = pd.DataFrame([
    # LIMIT_BAL, SEX, EDU, MAR, AGE, PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6,
    # BILL_AMT1..6, PAY_AMT1..6
    [50000,  2, 2, 1, 35,  0, 0, 0, 0, 0, 0, 3000, 2500, 2000, 1500, 1000,  500, 1000, 1000, 1000, 1000, 500,  500],  # perfil bajo riesgo
    [20000,  1, 3, 2, 28,  2, 2, 2, 2, 2, 2, 5000, 4800, 4600, 4400, 4200, 4000,    0,    0,    0,    0,   0,    0],  # perfil alto riesgo
    [200000, 2, 1, 1, 45, -1,-1,-1,-1,-1,-1, 8000, 7000, 6000, 5000, 4000, 3000, 8000, 7000, 6000, 5000, 4000, 3000], # perfil muy bajo riesgo
])

predicciones   = modelo.predict(perfiles)
probabilidades = modelo.predict_proba(perfiles)

etiquetas = ["Bajo riesgo (limite alto, pagos al dia)",
             "Alto riesgo (pagos atrasados, sin pagos)",
             "Muy bajo riesgo (limite muy alto, pagos completos)"]

print("Predicciones del modelo:")
print(f"  {'Perfil':<52} {'Pred':>6} {'P(default)':>12}")
print("  " + "-" * 72)
for etiqueta, pred, prob in zip(etiquetas, predicciones, probabilidades):
    resultado = "DEFAULT" if pred == 1 else "No default"
    print(f"  {etiqueta:<52} {resultado:>6}  {prob[1]*100:>9.1f}%")

---

**Resumen del notebook Azure ML Dia 4:**
- Conectaste al workspace de Azure ML con el SDK v2 y `DefaultAzureCredential`
- Escribiste un script de entrenamiento (`main.py`) con MLflow autolog
- Creaste un Compute Cluster que escala a 0 nodos cuando esta inactivo
- Lanzaste un Command Job con el dataset de credito bancario (30,000 clientes)
- Exploraste metricas, logs y el modelo en Azure ML Studio
- Descargaste el modelo y lo usaste localmente para predecir riesgo de default

**Dia 5:** Buenas practicas, IAM, gestion de costos y automatizacion en Azure.
Manana usaremos el dataset de programas sociales de Mexico.